In [ ]:
# Make sure Python can find the project modules
import sys, os

PROJECT_PATH = "smart_traffic_multi_agent"  # change if your folder has a different name
if os.path.exists(PROJECT_PATH):
    sys.path.append(PROJECT_PATH)
else:
    sys.path.append(".")  # fallback: assume notebook is already inside the project root

print("Python path updated. Ready to import project modules.")


## Step 1 — Import the real project modules (not re-typed code)

In [ ]:
from agents.traffic_agent import TrafficSignalAgent
from agents.central_agent import CentralControllerAgent
from agents.manager_agent import TrafficManagerAgent
from agents.communication import CommunicationLog
from simulation.simulation_engine import run_simulation, run_decentralized, build_agents
from simulation.traffic_generator import get_default_scenario
from decision.decision_engine import base_decision, classify_density

print("All modules imported successfully.")


## Step 2 — Create a single agent and inspect its decision

This demonstrates the core decision-making logic BEFORE any multi-agent
communication happens — just one junction, deciding its own green time.

In [ ]:
agent = TrafficSignalAgent(agent_id="Signal_1", vehicle_count=75, waiting_time=65)

print("Agent:", agent.agent_id)
print("Density:", agent.density)
print("Initial Green Time:", agent.green_time, "seconds")
print("Reason:", agent.reason)


## Step 3 — Load the default 4-junction demonstration scenario

In [ ]:
scenario = get_default_scenario()
for j in scenario:
    print(j)


## Step 4 — Centralized Architecture

One Central Controller sees ALL junctions and reallocates green time
across the whole network at once.

In [ ]:
result_centralized = run_simulation(scenario, "Centralized")

print("=== CENTRALIZED RESULTS ===")
for a in result_centralized["agents"]:
    print(f"{a.agent_id}: green_time={a.green_time}s (was {a.initial_green_time}s) | {a.reason}")

print("\nCommunication Log:")
for m in result_centralized["comm_log"]:
    print(f'[{m["timestamp"]}] {m["sender"]} -> {m["receiver"]} | {m["message_type"]} | status={m["status"]}')

print(f'\nAvg waiting before: {result_centralized["avg_waiting_before"]}s')
print(f'Avg waiting after:  {result_centralized["avg_waiting_after"]}s')
print(f'Improvement: {result_centralized["improvement_pct"]}%')


## Step 5 — Hierarchical Architecture

A Traffic Manager delegates to Area Managers, each optimizing only its
own group of signals.

In [ ]:
result_hierarchical = run_simulation(scenario, "Hierarchical")

print("=== HIERARCHICAL RESULTS ===")
for a in result_hierarchical["agents"]:
    print(f"{a.agent_id}: green_time={a.green_time}s (was {a.initial_green_time}s) | {a.reason}")

print(f'\nAvg waiting before: {result_hierarchical["avg_waiting_before"]}s')
print(f'Avg waiting after:  {result_hierarchical["avg_waiting_after"]}s')
print(f'Improvement: {result_hierarchical["improvement_pct"]}%')


## Step 6 — Decentralized Architecture (MAIN architecture of this project)

No controller at all — each signal negotiates directly with its peers.

In [ ]:
result_decentralized = run_simulation(scenario, "Decentralized")

print("=== DECENTRALIZED RESULTS ===")
for a in result_decentralized["agents"]:
    print(f"{a.agent_id}: green_time={a.green_time}s (was {a.initial_green_time}s) | {a.reason}")

print("\nCommunication Log:")
for m in result_decentralized["comm_log"]:
    print(f'[{m["timestamp"]}] {m["sender"]} -> {m["receiver"]} | {m["message_type"]} | status={m["status"]}')

print(f'\nAvg waiting before: {result_decentralized["avg_waiting_before"]}s')
print(f'Avg waiting after:  {result_decentralized["avg_waiting_after"]}s')
print(f'Improvement: {result_decentralized["improvement_pct"]}%')


## Step 7 — Emergency Vehicle Scenario

In [ ]:
emergency_scenario = get_default_scenario()
emergency_scenario[2]["emergency_vehicle"] = True  # Signal_3 has an emergency vehicle

result_emergency = run_simulation(emergency_scenario, "Decentralized")

print("=== EMERGENCY VEHICLE SCENARIO (Decentralized) ===")
for a in result_emergency["agents"]:
    flag = " <-- EMERGENCY" if a.emergency_vehicle else ""
    print(f"{a.agent_id}: green_time={a.green_time}s{flag} | {a.reason}")


## Step 8 — Compare All Three Architectures Side by Side

In [ ]:
import pandas as pd

rows = []
for arch, res in [("Centralized", result_centralized),
                   ("Hierarchical", result_hierarchical),
                   ("Decentralized", result_decentralized)]:
    rows.append({
        "Architecture": arch,
        "Avg Waiting Before (s)": res["avg_waiting_before"],
        "Avg Waiting After (s)": res["avg_waiting_after"],
        "Improvement (%)": res["improvement_pct"],
        "Messages Exchanged": len(res["comm_log"]),
    })

comparison_df = pd.DataFrame(rows)
comparison_df


## Step 9 — Error Handling Demonstration

The system validates inputs and fails with clear messages rather than
crashing silently.

In [ ]:
# Invalid vehicle count (negative)
try:
    bad_agent = TrafficSignalAgent(agent_id="Signal_X", vehicle_count=-10, waiting_time=20)
except ValueError as e:
    print("Caught expected error:", e)

# Invalid architecture name
try:
    run_simulation(scenario, "RandomArchitecture")
except ValueError as e:
    print("Caught expected error:", e)

# Empty junction data
try:
    run_simulation([], "Centralized")
except ValueError as e:
    print("Caught expected error:", e)
